# Embedding Model Benchmark — Feedback → Amendment Retrieval

**Research question**: Which sentence-transformer best retrieves an amendment targeting article N,
given a lobbying/stakeholder feedback submission that has had its explicit article references removed?

## Design

Post-proposal HYS feedback often cites article numbers directly. Amendments in the parliamentary
process target specific articles (`target_element`). This shared article reference creates a
**proxy gold standard** without requiring manual annotation:

| Step | What happens |
|------|--------------|
| 1 | Load post-proposal feedback chunks that contain exactly ONE `Article N` / `Recital N` citation |
| 2 | Strip that citation → decontextualized feedback |
| 3 | Build amendment corpus; link each amendment to its target article via `target_element` |
| 4 | Gold set for a query = all amendments targeting the same article the feedback cited |
| 5 | Embed both; measure first-relevant-document MRR and Recall\@K across 5 models |
| 6 | Flag near-paraphrase pairs (ROUGE-L) to separate lexical from semantic signal |

Multi-citation chunks (those mentioning more than one distinct article) are excluded — the
single-reference constraint ensures a clean, unambiguous gold label.

### Two evaluation variants

| Variant | Corpus embedding | Meaning |
|---------|------------------|---------|
| **pipeline** | `amended_text` only | **Real pipeline proxy** — mirrors `build_annotation_candidates.py` |
| **with_header** | `target_element + '\n' + amended_text` | Upper bound — structural cue present |

The **pipeline** variant is the one that matters for the thesis.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
PROCEDURE_ID = '2021/0106(COD)'

MODELS = [
    {
        'name':           'mpnet-base-v2',
        'hf_id':          'sentence-transformers/all-mpnet-base-v2',
        'query_prefix':   '',
        'passage_prefix': '',
    },
    {
        'name':           'bge-large-en-v1.5',
        'hf_id':          'BAAI/bge-large-en-v1.5',
        'query_prefix':   'Represent this sentence for searching relevant passages: ',
        'passage_prefix': '',
    },
    {
        'name':           'e5-large-v2',
        'hf_id':          'intfloat/e5-large-v2',
        'query_prefix':   'query: ',
        'passage_prefix': 'passage: ',
    },
    {
        'name':           'multilingual-e5-large',
        'hf_id':          'intfloat/multilingual-e5-large',
        'query_prefix':   'query: ',
        'passage_prefix': 'passage: ',
    },
    {
        'name':           'sbert-legal-xlm-roberta',
        'hf_id':          'Stern5497/sbert-legal-xlm-roberta-base',
        'query_prefix':   '',
        'passage_prefix': '',
    },
]

K_VALUES             = [1, 3, 5, 10]
MIN_CLEANED_LEN      = 80
PARAPHRASE_THRESHOLD = 0.30
MIN_GOLD_PAIRS       = 20

In [ ]:
import os, re, gc, json
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from dotenv import load_dotenv
from supabase import create_client
from sentence_transformers import SentenceTransformer

load_dotenv()
supabase = create_client(
    os.getenv('SUPABASE_URL'),
    os.getenv('SUPABASE_SERVICE_ROLE_KEY'),
)
Path('results').mkdir(exist_ok=True)
print('Connected to Supabase.')

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def paginate(table, select, filter_fn=None, page_size=1000):
    rows, offset = [], 0
    while True:
        q = supabase.table(table).select(select)
        if filter_fn:
            q = filter_fn(q)
        page = q.range(offset, offset + page_size - 1).execute().data or []
        rows.extend(page)
        if len(page) < page_size:
            break
        offset += page_size
    return rows


def embed_texts(model, texts, prefix='', batch_size=32):
    """Encode texts with optional prefix; returns L2-normalised (n, dim) float32 array."""
    if not texts:
        return np.empty((0, model.get_sentence_embedding_dimension()), dtype=np.float32)
    prefixed = [prefix + t for t in texts] if prefix else texts
    return model.encode(
        prefixed,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )


print('Helpers ready.')

## 1. Data Loading

In [ ]:
proc = (
    supabase.table('procedures')
    .select('title, proposal_date')
    .eq('id', PROCEDURE_ID)
    .single()
    .execute()
    .data
)
PROPOSAL_DATE = proc['proposal_date']
print(f'Procedure:     {PROCEDURE_ID}')
print(f'Title:         {proc["title"]}')
print(f'Proposal date: {PROPOSAL_DATE}')

In [ ]:
# ── Load amendments ───────────────────────────────────────────────────────────
amd_rows = paginate(
    'procedure_amendments',
    'id, amendment_number, target_element, target_type, original_text, amended_text, justification, committee',
    lambda q: q.eq('procedure_id', PROCEDURE_ID),
)
print(f'Amendments loaded: {len(amd_rows)}')

# Keep only amendments that target an article or recital and have usable text
amd_rows = [
    a for a in amd_rows
    if a.get('target_type') in ('article', 'recital')
    and len((a.get('amended_text') or a.get('original_text') or '').strip()) >= 80
]
print(f'With article/recital target and sufficient text: {len(amd_rows)}')

In [ ]:
# ── Load post-proposal feedback chunks ───────────────────────────────────────
fb_rows = paginate(
    'hys_feedback_chunks',
    'id, feedback_id, chunk_index, chunk_text, organisation_name, transparency_reg_id, date_feedback',
    lambda q: (
        q.eq('procedure_id', PROCEDURE_ID)
         .gte('date_feedback', PROPOSAL_DATE)
    ),
)
print(f'Post-proposal feedback chunks: {len(fb_rows)}')
print(f'Unique organisations:          {len({r.get("organisation_name") for r in fb_rows})}')

## 2. Gold Standard Construction

A **gold pair** links a feedback chunk to the set of amendments targeting the same article.

| Field | Source |
|-------|--------|
| `feedback_raw` | Original chunk (citations intact) |
| `feedback_clean` | Chunk with all `Article N` / `Recital N` refs stripped |
| `gold_amd_idxs` | Indices into `amd_rows` where `target_element` resolves to the same (type, number) |

Chunks citing more than one **distinct** (type, number) pair are excluded — the gold label
would be ambiguous.

In [ ]:
# ── Citation regex (identical to transformertest.ipynb) ───────────────────────
_NUM  = r'\(?\d+[a-z]?\)?'
_SEP  = r'(?:\s*(?:,|and|to|-)\s*' + _NUM + r')*'
_ART  = r'(?:articles?|art\.?)'
_REC  = r'(?:recitals?)'

STRIP_RE = re.compile(
    _ART + r'\s*' + _NUM + r'(?:,\s*paragraph\s*\d+[a-z]?)?' + _SEP
    + r'|' +
    _REC + r'\s*' + _NUM + _SEP,
    re.IGNORECASE,
)

_CAP_ART = re.compile(_ART + r'\s*\(?(\d+[a-z]?)\)?', re.IGNORECASE)
_CAP_REC = re.compile(_REC + r'\s*\(?(\d+)\)?',       re.IGNORECASE)


def clean_citations(text):
    out = STRIP_RE.sub('', text)
    out = re.sub(r'[ \t]{2,}', ' ', out)
    out = re.sub(r'\n{3,}', '\n\n', out)
    return out.strip()


def citation_window(text, match, window=900):
    start = max(0, match.start() - window)
    end   = min(len(text), match.end() + window)
    return text[start:end]


# ── Amendment target_element normalisation ────────────────────────────────────
_AMT_ART = re.compile(r'article\s*\(?(\d+[a-z]?)\)?', re.IGNORECASE)
_AMT_REC = re.compile(r'recital\s*\(?(\d+[a-z]?)\)?', re.IGNORECASE)

def parse_amd_target(target_element):
    """Extract (type, normalised_number) from a target_element string.

    Examples:
      'Article 5(2)'  -> ('article', '5')
      'Recital 12'    -> ('recital', '12')
      'Article 5a'    -> ('article', '5a')
    Only the top-level article number is used; paragraph/sub-paragraph are ignored.
    """
    if not target_element:
        return None, None
    m = _AMT_ART.search(target_element)
    if m:
        return 'article', m.group(1).lower().lstrip('0') or '0'
    m = _AMT_REC.search(target_element)
    if m:
        return 'recital', m.group(1).lower().lstrip('0') or '0'
    return None, None


# Build amendment index: (type, number) -> list of row indices
amd_key_to_idxs = defaultdict(list)
for idx, a in enumerate(amd_rows):
    etype, enum = parse_amd_target(a.get('target_element', ''))
    if etype and enum:
        amd_key_to_idxs[(etype, enum)].append(idx)

print(f'Distinct article/recital targets in amendments: {len(amd_key_to_idxs)}')
print(f'Mean amendments per target: {np.mean([len(v) for v in amd_key_to_idxs.values()]):.1f}')
print(f'Max amendments per target:  {max(len(v) for v in amd_key_to_idxs.values())}')

In [ ]:
# ── Chunk lookup for context windows ─────────────────────────────────────────
chunk_lookup = {
    (r['feedback_id'], r['chunk_index']): r['chunk_text']
    for r in fb_rows
}

# ── Build gold pairs ──────────────────────────────────────────────────────────
gold_pairs = []
skipped    = defaultdict(int)
seen_keys  = set()  # deduplicate (chunk_id, element) for repeated citations

for row in fb_rows:
    current = (row.get('chunk_text') or '').strip()
    if not current:
        skipped['empty_chunk'] += 1
        continue

    nxt      = chunk_lookup.get((row['feedback_id'], row['chunk_index'] + 1), '')
    combined = (current + ' ' + nxt).strip() if nxt else current

    matches = (
        [('article', m) for m in _CAP_ART.finditer(combined)] +
        [('recital', m) for m in _CAP_REC.finditer(combined)]
    )
    if not matches:
        skipped['no_citation'] += 1
        continue

    # Resolve all cited (type, number) pairs
    cited = set()
    for etype, match in matches:
        enum = match.group(1).lower().lstrip('0') or '0'
        cited.add((etype, enum))

    # Exclude multi-citation chunks
    if len(cited) > 1:
        skipped['multi_citation'] += 1
        continue

    etype, enum = next(iter(cited))

    # Must have at least one amendment targeting the cited article
    gold_idxs = amd_key_to_idxs.get((etype, enum), [])
    if not gold_idxs:
        skipped['no_amendment_for_article'] += 1
        continue

    # Deduplicate: same chunk should not produce multiple pairs
    key = (row['id'], etype, enum)
    if key in seen_keys:
        continue
    seen_keys.add(key)

    # Use the citation match for a context window
    first_match = matches[0][1]
    window_text = citation_window(combined, first_match, window=900)
    cleaned     = clean_citations(window_text)

    if len(cleaned) < MIN_CLEANED_LEN:
        skipped['too_short_after_clean'] += 1
        continue

    gold_pairs.append({
        'chunk_id':       row['id'],
        'org':            row.get('organisation_name'),
        'feedback_raw':   window_text,
        'feedback_clean': cleaned,
        'element_type':   etype,
        'element_number': enum,
        'gold_amd_idxs':  gold_idxs,
        'n_gold':         len(gold_idxs),
    })

print(f'Gold pairs:                {len(gold_pairs)}')
print(f'Unique articles cited:     {len({(p["element_type"], p["element_number"]) for p in gold_pairs})}')
print(f'Unique feedback chunks:    {len({p["chunk_id"] for p in gold_pairs})}')
print(f'Mean amendments per pair:  {np.mean([p["n_gold"] for p in gold_pairs]):.1f}')
print(f'Skipped: {dict(skipped)}')

if len(gold_pairs) < MIN_GOLD_PAIRS:
    raise ValueError(
        f'Only {len(gold_pairs)} gold pairs (need >= {MIN_GOLD_PAIRS}).\n'
        'Try a procedure with more post-proposal feedback, e.g. 2021/0106(COD).'
    )

gold_df = pd.DataFrame(gold_pairs)

In [ ]:
# ── Citation density filter (same logic as transformertest) ───────────────────
from collections import Counter

cites_per_chunk = Counter(p['chunk_id'] for p in gold_pairs)
noisy_chunks    = {cid for cid, n in cites_per_chunk.items() if n >= 5}
before          = len(gold_pairs)
gold_pairs      = [p for p in gold_pairs if p['chunk_id'] not in noisy_chunks]
print(f'Density filter: dropped {before - len(gold_pairs)} pairs → {len(gold_pairs)} remaining')

# ── Paraphrase detection via ROUGE-L ─────────────────────────────────────────
def _lcs_len(a, b):
    a, b = a[:150], b[:150]
    m, n = len(a), len(b)
    if not m or not n:
        return 0
    prev = [0] * (n + 1)
    for i in range(m):
        curr = [0] * (n + 1)
        for j in range(n):
            curr[j + 1] = prev[j] + 1 if a[i] == b[j] else max(prev[j + 1], curr[j])
        prev = curr
    return prev[n]


def rouge_l_f1(hyp, ref):
    h = hyp.lower().split()
    r = ref.lower().split()
    if not h or not r:
        return 0.0
    lcs  = _lcs_len(h, r)
    prec = lcs / len(h)
    rec  = lcs / len(r)
    return 2 * prec * rec / (prec + rec) if prec + rec else 0.0


# For multi-label: take the max ROUGE-L against any gold amendment text
def max_rouge_l(feedback_clean, gold_idxs):
    best = 0.0
    for idx in gold_idxs:
        amd_text = (amd_rows[idx].get('amended_text') or amd_rows[idx].get('original_text') or '')
        best = max(best, rouge_l_f1(feedback_clean, amd_text))
    return best


gold_df = pd.DataFrame(gold_pairs)
gold_df['rouge_l']       = [max_rouge_l(p['feedback_clean'], p['gold_amd_idxs']) for p in gold_pairs]
gold_df['is_paraphrase'] = gold_df['rouge_l'] >= PARAPHRASE_THRESHOLD

print(f'Near-paraphrases (ROUGE-L >= {PARAPHRASE_THRESHOLD}): '
      f'{gold_df["is_paraphrase"].mean():.1%}  '
      f'({gold_df["is_paraphrase"].sum()} / {len(gold_df)})')
print(f'ROUGE-L  mean={gold_df["rouge_l"].mean():.3f}  '
      f'median={gold_df["rouge_l"].median():.3f}  '
      f'max={gold_df["rouge_l"].max():.3f}')

In [ ]:
# ── Dataset summary ───────────────────────────────────────────────────────────
print('=== Gold Standard (Feedback → Amendment) ===')
print(f'Procedure:               {PROCEDURE_ID}')
print(f'Amendment corpus size:   {len(amd_rows)}')
print(f'Gold pairs total:        {len(gold_pairs)}')
print(f'  Semantic (non-para):   {(~gold_df["is_paraphrase"]).sum()}')
print(f'  Near-paraphrase:       {gold_df["is_paraphrase"].sum()}')
unique_cited = gold_df[['element_type', 'element_number']].drop_duplicates()
print(f'  Unique articles cited: {len(unique_cited)} of {len(amd_key_to_idxs)} with amendments')
print(f'  Mean gold set size:    {gold_df["n_gold"].mean():.1f} amendments per pair')
print()
print('Top 15 most-cited elements:')
cit_dist = gold_df.groupby(['element_type', 'element_number']).size().sort_values(ascending=False)
print(cit_dist.head(15).to_string())

# ── Amendment corpus arrays ───────────────────────────────────────────────────
# pipeline variant: amended_text (or original_text) only — mirrors build_annotation_candidates.py
# with_header variant: target_element prepended — upper bound
amd_text_only = [
    (a.get('amended_text') or a.get('original_text') or '').strip()
    for a in amd_rows
]
amd_with_header = [
    (a.get('target_element') or '') + '\n' + (a.get('amended_text') or a.get('original_text') or '').strip()
    for a in amd_rows
]

# ── Query arrays ─────────────────────────────────────────────────────────────
feedback_raw   = [p['feedback_raw']   for p in gold_pairs]
feedback_clean = [p['feedback_clean'] for p in gold_pairs]
gold_amd_idxs  = [p['gold_amd_idxs'] for p in gold_pairs]  # list of lists

print(f'\nQuery set:    {len(feedback_clean)} gold pairs')
print(f'gold_df rows: {len(gold_df)}  (must match above)')

## 3. Embedding Evaluation

For each model we embed all corpus/query pools once, compute two similarity matrices
(one per variant), and record first-relevant-document MRR and Recall\@K.

Because the gold label is a **set** of amendments (not a single index), metrics are:
- **Recall\@K**: 1 if ANY gold amendment appears in the model's top-K for this query
- **MRR**: 1 / rank of the first gold amendment in the ranked list

In [ ]:
# Variant: (name, query_pool_key, corpus_pool_key, description)
VARIANTS = [
    ('pipeline',    'fb_clean', 'amd_text',   'Clean query + text only — real pipeline proxy'),
    ('with_header', 'fb_clean', 'amd_header', 'Clean query + target_element header — upper bound'),
]

all_results  = []  # one dict per (model, variant)
per_query_db = {}  # (model, variant) -> list of per-query dicts
sim_db       = {}  # (model, variant) -> (n_queries, n_amendments) sim matrix

for cfg in MODELS:
    name, hf_id = cfg['name'], cfg['hf_id']
    qpfx, ppfx  = cfg['query_prefix'], cfg['passage_prefix']

    print(f'\n{"="*62}')
    print(f'  {name}  ({hf_id})')
    print(f'{"="*62}')

    mdl = SentenceTransformer(hf_id)

    print('  [1/3] amendment corpus (text only)  ...')
    emb_amd_text   = embed_texts(mdl, amd_text_only,   prefix=ppfx)
    print('  [2/3] amendment corpus (with header)...')
    emb_amd_header = embed_texts(mdl, amd_with_header, prefix=ppfx)
    print('  [3/3] feedback queries (clean)      ...')
    emb_fb_clean   = embed_texts(mdl, feedback_clean,  prefix=qpfx)

    del mdl; gc.collect()

    pool = {
        'amd_text':   emb_amd_text,
        'amd_header': emb_amd_header,
        'fb_clean':   emb_fb_clean,
    }

    for v_name, q_key, a_key, _ in VARIANTS:
        sim = pool[q_key] @ pool[a_key].T   # (n_queries, n_amendments)
        sim_db[(name, v_name)] = sim

        per_q, rr = [], []

        for qi, gold_set in enumerate(gold_amd_idxs):
            scores = sim[qi]
            ranked = np.argsort(scores)[::-1]   # best first

            # Rank of first gold amendment (first-relevant-document MRR)
            first_hit_rank = None
            for pos, amd_idx in enumerate(ranked):
                if amd_idx in gold_set:
                    first_hit_rank = pos + 1
                    break

            rank = first_hit_rank if first_hit_rank is not None else len(ranked) + 1
            rr.append(1.0 / rank)

            # Score of the best gold amendment
            best_gold_score = float(max(scores[i] for i in gold_set))

            per_q.append({
                'rank':            rank,
                'n_gold':          len(gold_set),
                'score_best_gold': best_gold_score,
                'score_top1':      float(scores[ranked[0]]),
                'top1_amd_idx':    int(ranked[0]),
                'rouge_l':         gold_df.iloc[qi]['rouge_l'],
                'is_paraphrase':   bool(gold_df.iloc[qi]['is_paraphrase']),
            })

        per_query_db[(name, v_name)] = per_q

        row = {'model': name, 'variant': v_name, 'mrr': float(np.mean(rr))}
        for k in K_VALUES:
            # R@K: any gold amendment in top-K
            row[f'R@{k}'] = sum(
                1 for qi, gold_set in enumerate(gold_amd_idxs)
                if any(idx in gold_set for idx in np.argsort(sim[qi])[::-1][:k])
            ) / len(gold_amd_idxs)
        all_results.append(row)

        print(f'  [{v_name:12s}]  MRR={row["mrr"]:.3f}  ' +
              '  '.join(f'R@{k}={row[f"R@{k}"]:.0%}' for k in K_VALUES))

print('\nAll models evaluated.')

## 4. Results

In [ ]:
results_df = pd.DataFrame(all_results)

for v_name, _, _, desc in VARIANTS:
    sub = (
        results_df[results_df['variant'] == v_name]
        .sort_values('mrr', ascending=False)
        .drop(columns='variant')
        .set_index('model')
    )
    disp = sub.copy()
    for col in disp.columns:
        disp[col] = disp[col].map('{:.1%}'.format)
    print(f'\n── {v_name.upper()} ({desc}) ──')
    print(disp.to_string())

pipeline_rank = (
    results_df[results_df['variant'] == 'pipeline']
    .sort_values('mrr', ascending=False)
    [['model', 'mrr'] + [f'R@{k}' for k in K_VALUES]]
    .reset_index(drop=True)
)
pipeline_rank.index += 1
print('\n── RANKING on PIPELINE variant ──')
print(pipeline_rank.to_string())

In [ ]:
# ── Figure 1: Recall@k and MRR comparison ────────────────────────────────────
pipeline_df = results_df[results_df['variant'] == 'pipeline']
model_order = pipeline_df.sort_values('mrr', ascending=False)['model'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Feedback → Amendment Benchmark — {PROCEDURE_ID}', fontsize=13, fontweight='bold')

# Left: Recall@k (pipeline variant)
ax    = axes[0]
x     = np.arange(len(K_VALUES))
width = 0.8 / len(MODELS)

for i, m in enumerate(model_order):
    row  = pipeline_df[pipeline_df['model'] == m].iloc[0]
    vals = [row[f'R@{k}'] for k in K_VALUES]
    ax.bar(x + i * width, vals, width, label=m)

ax.set_xticks(x + width * (len(MODELS) - 1) / 2)
ax.set_xticklabels([f'R@{k}' for k in K_VALUES])
ax.set_ylabel('Recall')
ax.set_title('Recall@k  (pipeline variant — text-only amendment corpus)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Right: MRR across both variants
ax2    = axes[1]
x2     = np.arange(len(model_order))
w2     = 0.35
colors = ['#4878CF', '#D65F5F']

for i, (v_name, _, _, _) in enumerate(VARIANTS):
    v_df = results_df[results_df['variant'] == v_name]
    mrrs = [v_df[v_df['model'] == m]['mrr'].values[0] for m in model_order]
    ax2.bar(x2 + i * w2, mrrs, w2, label=v_name, color=colors[i])

ax2.set_xticks(x2 + w2 / 2)
ax2.set_xticklabels(model_order, rotation=15, ha='right', fontsize=9)
ax2.set_ylabel('MRR')
ax2.set_title('MRR: pipeline vs with_header')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax2.legend(title='variant')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/transformer_benchmark_forward.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> results/transformer_benchmark_forward.png')

In [ ]:
# ── Figure 2: Paraphrase vs semantic breakdown ────────────────────────────────
best_model = model_order[0]
qdata_df   = pd.DataFrame(per_query_db[(best_model, 'pipeline')])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Paraphrase Analysis — {best_model} (pipeline variant)', fontsize=12, fontweight='bold')

ax = axes[0]
x  = np.arange(len(K_VALUES))
w  = 0.35

for offset, flag, label, color in [
    (0, False, f'Semantic  (n={(~qdata_df["is_paraphrase"]).sum()})', '#4878CF'),
    (w, True,  f'Near-para (n={qdata_df["is_paraphrase"].sum()})',   '#D65F5F'),
]:
    sub     = qdata_df[qdata_df['is_paraphrase'] == flag]
    recalls = [(sub['rank'] <= k).mean() if len(sub) else 0.0 for k in K_VALUES]
    ax.bar(x + offset, recalls, w, label=label, color=color)

ax.set_xticks(x + w / 2)
ax.set_xticklabels([f'R@{k}' for k in K_VALUES])
ax.set_ylabel('Recall (first-hit)')
ax.set_title('Recall@k: semantic vs near-paraphrase pairs')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
ax2.hist(qdata_df['score_best_gold'], bins=30, alpha=0.6, label='Best gold amendment', color='#4878CF')
ax2.hist(qdata_df['score_top1'],      bins=30, alpha=0.6, label='Top-1 retrieved',     color='#D65F5F')
ax2.set_xlabel('Cosine similarity')
ax2.set_ylabel('Count')
ax2.set_title('Score: best gold amendment vs top-1 retrieved')
ax2.legend(fontsize=8)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/paraphrase_analysis_forward.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> results/paraphrase_analysis_forward.png')

# Semantic-only table
print('\n── Semantic-only metrics (non-paraphrase, pipeline variant) ──')
sem_rows = []
for cfg in MODELS:
    m  = cfg['name']
    qd = pd.DataFrame(per_query_db[(m, 'pipeline')])
    sem = qd[~qd['is_paraphrase']]
    if len(sem) == 0:
        continue
    row = {'model': m, 'n': len(sem), 'MRR': float((1 / sem['rank']).mean())}
    for k in K_VALUES:
        row[f'R@{k}'] = float((sem['rank'] <= k).mean())
    sem_rows.append(row)

sem_df = pd.DataFrame(sem_rows).sort_values('MRR', ascending=False)
print(sem_df.to_string(index=False))

## 5. Interpretation Guide

### Metrics

Because each query has a **set** of gold amendments (all those targeting the cited article),
the metrics are first-relevant-document:

| Metric | Meaning |
|--------|---------|
| **MRR** | Mean 1/rank of the first gold amendment retrieved |
| **R@K** | Fraction of queries where at least one gold amendment is in top-K |

### Reading the variant gap

- **pipeline → with_header** gap: how much the model benefits from seeing `target_element`
  in the corpus. A large gap means the model uses the structural header as a shortcut —
  this inflates apparent performance but doesn't reflect the real pipeline task.
  Report **pipeline** metrics in the thesis.

### Gold set size effect

A large mean gold set (many amendments per article) makes R@K look higher by chance —
there are more ways to get a hit. Check `gold_df['n_gold'].describe()` and consider
reporting a **normalised** MRR that controls for gold set size if this is large.

### Choosing a cosine threshold

The score distribution plot (Figure 2, right) shows where `score_best_gold` and `score_top1`
diverge. Pairs where `score_best_gold < score_top1` are rank > 1 failures. Pick
`MIN_SCORE` at the point where the histograms separate — the same `0.84` from the
article benchmark may not transfer here.

## 6. Qualitative Error Analysis

Manual inspection of best hits, hard failures, and a score-stratified sample for the
best-performing model (pipeline variant).

For each case we show:
- The feedback (citations stripped)
- Which article/recital it cited — and how many amendments target that element
- The **best gold amendment** (highest cosine similarity among the gold set)
- The **top-1 retrieved amendment** (only shown when it is not a gold amendment)

In [ ]:

# ── Qualitative error analysis — best model, pipeline variant ─────────────────
TRUNC_FB  = 600
TRUNC_AMD = 350

best_model = model_order[0]
qdata      = per_query_db[(best_model, 'pipeline')]
sim_mat    = sim_db[(best_model, 'pipeline')]   # (n_queries, n_amendments)


def _best_gold_amd_idx(qi):
    """Index of the gold amendment with the highest cosine score for query qi."""
    gold_set = gold_amd_idxs[qi]
    return max(gold_set, key=lambda i: sim_mat[qi, i])


def _sample_near_percentile(pct):
    scores = [d['score_best_gold'] for d in qdata]
    target = float(np.percentile(scores, pct))
    return min(range(len(qdata)), key=lambda i: abs(qdata[i]['score_best_gold'] - target))


def print_case(label, qi, d):
    row      = gold_df.iloc[qi]
    sep      = '─' * 72

    gold_elem = f"{row['element_type'].capitalize()} {row['element_number']}"
    n_gold    = d['n_gold']

    # Best gold amendment
    best_gold_idx = _best_gold_amd_idx(qi)
    best_gold_amd = amd_rows[best_gold_idx]

    # Top-1 retrieved
    top1_idx = d['top1_amd_idx']
    top1_amd = amd_rows[top1_idx]
    top1_is_gold = top1_idx in set(gold_amd_idxs[qi])

    print(f'\n{sep}')
    print(f'  {label}')
    print(f'  Rank {d["rank"]:>4}  |  score_best_gold={d["score_best_gold"]:.4f}  '
          f'score_top1={d["score_top1"]:.4f}  ROUGE-L={d["rouge_l"]:.3f}')
    print(f'  Gold element: {gold_elem}  ({n_gold} amendments target this element)')
    print(f'  Org: {row["org"] or "—"}')

    print(f'\n  FEEDBACK (citations stripped):')
    print(f'  {row["feedback_clean"][:TRUNC_FB]!r}')

    print(f'\n  BEST GOLD AMENDMENT  [{best_gold_amd.get("target_element", "?")}]'
          f'  amd#{best_gold_amd.get("amendment_number", "?")}  '
          f'score={sim_mat[qi, best_gold_idx]:.4f}:')
    best_text = (best_gold_amd.get('amended_text') or best_gold_amd.get('original_text') or '')
    print(f'  {best_text[:TRUNC_AMD]!r}')
    if best_gold_amd.get('justification'):
        print(f'  justification: {best_gold_amd["justification"][:200]!r}')

    if not top1_is_gold:
        print(f'\n  TOP-1 RETRIEVED (wrong)  [{top1_amd.get("target_element", "?")}]'
              f'  amd#{top1_amd.get("amendment_number", "?")}  '
              f'score={d["score_top1"]:.4f}:')
        top1_text = (top1_amd.get('amended_text') or top1_amd.get('original_text') or '')
        print(f'  {top1_text[:TRUNC_AMD]!r}')
    else:
        print(f'\n  TOP-1 RETRIEVED: ✓ is a gold amendment')

    print(sep)


# ── 1. Best hits ──────────────────────────────────────────────────────────────
print('=' * 72)
print(f'  BEST HITS — {best_model}  (rank=1, highest score_best_gold)')
print('=' * 72)

best_hits = sorted(
    [(i, d) for i, d in enumerate(qdata) if d['rank'] == 1],
    key=lambda x: -x[1]['score_best_gold'],
)
for qi, d in best_hits[:3]:
    print_case('BEST HIT', qi, d)

# ── 2. Hard failures ──────────────────────────────────────────────────────────
print('\n' + '=' * 72)
print(f'  HARD FAILURES — rank > 10, highest score_best_gold')
print('=' * 72)

hard_fail = sorted(
    [(i, d) for i, d in enumerate(qdata) if d['rank'] > 10],
    key=lambda x: -x[1]['score_best_gold'],
)
for qi, d in hard_fail[:3]:
    print_case('HARD FAILURE', qi, d)

# ── 3. Score-stratified sample ────────────────────────────────────────────────
print('\n' + '=' * 72)
print(f'  STRATIFIED SAMPLE — one case per score_best_gold percentile')
print('=' * 72)

for pct in [10, 30, 50, 70, 90]:
    qi = _sample_near_percentile(pct)
    d  = qdata[qi]
    print_case(f'p{pct}  (score_best_gold≈{d["score_best_gold"]:.4f})', qi, d)


In [ ]:
# ── Save results ──────────────────────────────────────────────────────────────
proc_slug = PROCEDURE_ID.replace('/', '_').replace('(', '').replace(')', '')
out_csv   = Path('results') / f'transformer_benchmark_forward_{proc_slug}.csv'
out_json  = Path('results') / f'transformer_benchmark_forward_{proc_slug}.json'

results_df.to_csv(out_csv, index=False)

export = {
    'procedure_id':    PROCEDURE_ID,
    'n_gold_pairs':    len(gold_pairs),
    'n_amendments':    len(amd_rows),
    'paraphrase_rate': float(gold_df['is_paraphrase'].mean()),
    'mean_gold_size':  float(gold_df['n_gold'].mean()),
    'aggregate':       results_df.to_dict(orient='records'),
    'per_query':       {
        f'{m}__{v}': per_query_db[(m, v)]
        for m, v in per_query_db
    },
}
with open(out_json, 'w', encoding='utf-8') as fh:
    json.dump(export, fh, indent=2)

print(f'Saved CSV  -> {out_csv}')
print(f'Saved JSON -> {out_json}')